# Task 2 ensemble: which of two books did the user read?

Validation pairs are built like the test pairs: a held-out interaction against the book of a random other interaction
(drawn in proportion to popularity). Sub-models score (user, book); a LightGBM pair blender decides every pair.
The Task 2 test file has no timestamp.

- **popularity**: log(1 + #interactions of the book)
- **item cosine**: similarity of the book to the books the user has read
- **EASE**: linear autoencoder in closed form (Steck 2019), all books, computed in RAM
- **Mult-VAE**: non-linear variational autoencoder (Liang et al. 2018), trained on the GPU with early stopping

Every sub-model cell stores its scores in `assignment1_2026/trained_models/task2/<model>.pt`; the blend cells only need the stored files.
All hyperparameters: `lib/config.py`, Task 2 section.

In [ ]:
%load_ext autoreload
%autoreload 2

# assignment1.py and lib/ live next to this notebook (in submission/)
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

import torch

import assignment1
from lib.config import TASK2_PAIR_BLENDER
from lib.models.ease_model import load_ease_model, load_time_ease_model, store_ease_model, store_time_ease_model
from lib.models.item_cosine_model import load_item_cosine_model, store_item_cosine_model
from lib.models.mult_vae import load_mult_vae, store_mult_vae
from lib.models.pair_blender import load_pair_blender, store_pair_blender
from lib.models.popularity_model import load_popularity_model, store_popularity_model

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

### Data and validation pairs
Interactions $(u, i)$ of the training file. Each test row: user $u$ and two books, one of them read by $u$ (the real one $i^+$), one drawn as a negative $i^-$.
Validation pairs are built the same way: 10 % of the interactions are held out as $i^+$, the negative is the book of a random other interaction
(so $P(i^- = i) \propto |U_i|$, popularity-weighted, regardless of time), random order. Every sub-model gives a score $s(u, i)$:
$$\text{accuracy} = \frac{1}{|\mathcal P|}\sum_{(u,i^+,i^-)\in\mathcal P}\big[s(u,i^+) > s(u,i^-)\big]$$

In [ ]:
# training file -> validation pairs (held-out interactions vs. popularity-drawn negatives) + the test pairs
task2 = assignment1.load_task2()

## Sub-models

### Popularity
$$s(u,i) = \log\big(1 + |U_i|\big)$$
$U_i$ = users who read $i$. On its own close to a coin flip here (the negatives are drawn in proportion to popularity), but the blender uses it to correct the popularity bias of the other scores.

In [ ]:
# popularity: log(1 + #interactions of the book) (seconds)
popularity_result = assignment1.train_task2_popularity_model(task2)
store_popularity_model(popularity_result, "task2")

### Item cosine similarity to the user's history (PML Ch. 4)
$$\mathrm{sim}(i,j) = \frac{|U_i\cap U_j|}{\sqrt{|U_i|\,|U_j|}}\cdot\frac{n_{ij}}{n_{ij}+\lambda}, \qquad n_{ij} = |U_i\cap U_j|$$
$$s(u,i) = \sum_{j\in I_u\cap N_k(i)}\mathrm{sim}(i,j)$$
$N_k(i)$ = the $k = 100$ most similar books of $i$; $I_u$ = the books $u$ has read.

In [ ]:
# item cosine similarity to the user's history (CPU, about a minute)
item_cosine_result = assignment1.train_task2_item_cosine_model(task2)
store_item_cosine_model(item_cosine_result, "task2")

### EASE: linear autoencoder in closed form (Steck 2019)
$X$ = user x book 0/1 matrix.
$$\hat B = \arg\min_B\ \lVert X - XB\rVert_F^2 + \lambda\lVert B\rVert_F^2 \quad \text{s.t.}\ \operatorname{diag}(B) = 0$$
$$P = \big(X^\top X + \lambda I\big)^{-1}, \qquad \hat B_{ij} = -\frac{P_{ij}}{P_{jj}}\ (i\neq j), \qquad \hat B_{jj} = 0$$
$$s(u,i) = \sum_{j\in I_u}\hat B_{ji} = \big(X\hat B\big)_{ui}$$
$\lambda = 500$; books outside the `max_items` most popular ones get no weights (score missing).

In [ ]:
# EASE: one Gram matrix + one matrix inverse per fit (all ~51k books in RAM, a few minutes per fit); only the scores are stored, not the dense weights
ease_result = assignment1.train_task2_ease_model(task2)
store_ease_model(ease_result, "task2")

### Mult-VAE: variational autoencoder (Liang et al. 2018)
$\mathbf x_u$ = the user's 0/1 history over the books. Encoder ($W_k$ = affine layers):
$$\big[\boldsymbol\mu_u,\ \log\boldsymbol\sigma_u^2\big] = W_2\tanh\Big(W_1\,\mathrm{dropout}\big(\mathbf x_u/\lVert\mathbf x_u\rVert\big)\Big), \qquad \mathbf z_u = \boldsymbol\mu_u + \boldsymbol\sigma_u\odot\boldsymbol\epsilon,\ \ \boldsymbol\epsilon\sim\mathcal N(0, I)$$
Decoder and loss (multinomial likelihood + KL term, $\beta$ raised linearly from 0 to 0.2 over 20 epochs):
$$\boldsymbol\pi_u = \operatorname{softmax}\big(W_4\tanh(W_3\,\mathbf z_u)\big), \qquad \mathcal L(u) = -\sum_i x_{ui}\log\pi_{ui} + \beta\,\mathrm{KL}\big(\mathcal N(\boldsymbol\mu_u, \boldsymbol\sigma_u^2)\,\Vert\,\mathcal N(0, I)\big)$$
Score: the logit with $\mathbf z_u = \boldsymbol\mu_u$, i.e. $s(u,i) = \big(W_4\tanh(W_3\boldsymbol\mu_u)\big)_i$. Sizes 600 / 200, dropout 0.5;
early stopping on the validation pair accuracy, then a refit on all interactions.

In [ ]:
# Mult-VAE: non-linear autoencoder, early stopping on the validation pair accuracy, then refit on all interactions (GPU)
mult_vae_result = assignment1.train_task2_mult_vae(task2)
store_mult_vae(mult_vae_result, "task2")

## Blend

### Blender input
For every validation pair: the scores $s^{(m)}(u, i_1)$, $s^{(m)}(u, i_2)$ of both books from every sub-model in `combined_models` (models that never saw the held-out interaction).

In [ ]:
# stored sub-models of the blend (also works right after a kernel restart)
sub_model_loaders = {
    "popularity": load_popularity_model,
    "item_cosine": load_item_cosine_model,
    "ease": load_ease_model,
    "time_ease": load_time_ease_model,
    "mult_vae": load_mult_vae,
}
task2_model_results = {model_name: sub_model_loaders[model_name]("task2") for model_name in TASK2_PAIR_BLENDER.combined_models}

### Pair blender (LightGBM classifier)
Features of book $k$ of a pair: $\mathbf f_k = \big[s^{(1)}(u,i_k), \dots, s^{(M)}(u,i_k),\ [i_k \in I_u]\big]$ (+ time features where available). Features of the pair in a given order:
$$\mathbf x_{12} = \big[\mathbf f_1 - \mathbf f_2,\ \mathbf f_1,\ \mathbf f_2,\ \log(1+|I_u|)\big]$$
$g$ = gradient-boosted trees with log loss, trained on every validation pair in both orders ($\mathbf x_{12}$ with label $y$, $\mathbf x_{21}$ with $1-y$):
$$P(i_1\ \text{is the real book}) = \tfrac12\Big(g(\mathbf x_{12}) + 1 - g(\mathbf x_{21})\Big)$$
Score: 5-fold out-of-fold accuracy on the validation pairs. Prediction: item1 if the probability is above 0.5.

In [ ]:
# LightGBM pair blender: P(item1 is the real one) from both books' scores + meta-features (prints the validation accuracies)
pair_blend_result = assignment1.train_task2_blender(task2, task2_model_results)
store_pair_blender(pair_blend_result, "task2")

### Submission
`which` = item1 where $P(i_1\ \text{is the real book}) > 0.5$, otherwise item2.

In [ ]:
# writes output/task_2_predictions.csv (the "which" column = the chosen book)
submitted_blend_result = load_pair_blender("task2")
assignment1.write_task2_predictions(submitted_blend_result.test_predictions)
print("wrote", assignment1.TASK2_PREDICTION_FILE)